# MLOps Training 2026/2027 — Task 2

## Notebook 2 — Create the Labels

This notebook reads the order-level artifact from Notebook 1, creates the binary `is_late` target from actual versus estimated delivery dates, validates several real orders, checks class distribution, and saves the labeled artifact.

**Input:** `artifacts/01_joined_ml_table.parquet`

**Output:** `artifacts/02_labeled_table.parquet`


## 1. Load and prepare the order-level table

The label is defined only for delivered orders with both required dates. Date columns are parsed explicitly because the source database stored them as text.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\HP\Desktop\MLOps")
TASK2_ROOT = PROJECT_ROOT / "Task2"
ARTIFACTS_DIR = TASK2_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

input_path = ARTIFACTS_DIR / "tables" / "01_joined_ml_table.parquet"
output_path = ARTIFACTS_DIR / "tables" / "02_labeled_table.parquet"
metadata_path = ARTIFACTS_DIR / "reports" / "02_labeled_table_metadata.json"
if not input_path.exists():
    raise FileNotFoundError(input_path)

df = pd.read_parquet(input_path)
for col in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

required_dates = ["order_delivered_customer_date", "order_estimated_delivery_date"]
work = df[(df["order_status"] == "delivered") & df[required_dates].notna().all(axis=1)].copy()
work["is_late"] = (work["order_delivered_customer_date"] > work["order_estimated_delivery_date"]).astype("int8")
work = work.sort_values("order_purchase_timestamp").reset_index(drop=True)
print(f"Input rows: {len(df):,}")
print(f"Labeled delivered rows: {len(work):,}")
display(work[["order_id", "order_status", "order_delivered_customer_date", "order_estimated_delivery_date", "is_late"]].head())


Input rows: 99,441
Labeled delivered rows: 96,470


,order_id,order_status,order_delivered_customer_date,order_estimated_delivery_date,is_late
0,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-11-09 07:47:38,2016-10-04,1
1,3b697a20d9e427646d92567910af6d57,delivered,2016-10-26 14:02:13,2016-10-27,0
2,be5bc2f0da14d8071e2d45451ad119d9,delivered,2016-10-27 18:19:38,2016-11-07,0
3,a41c8759fbe7aab36ea07e038b2d4465,delivered,2016-11-03 10:58:07,2016-11-29,0
4,d207cc272675637bfed0062edffd0818,delivered,2016-10-31 11:07:42,2016-11-23,0


## 2. Validate the label on real orders

The following sample includes late and on-time orders when both classes are available. The expected rule is visible beside each row.


In [2]:
sample_parts = []
for label in [0, 1]:
    part = work[work["is_late"] == label].head(3)
    sample_parts.append(part)
label_sample = pd.concat(sample_parts).drop_duplicates("order_id")
label_sample = label_sample[["order_id", "order_delivered_customer_date", "order_estimated_delivery_date", "is_late"]]
label_sample["rule_check"] = (label_sample["order_delivered_customer_date"] > label_sample["order_estimated_delivery_date"]).astype("int8")
display(label_sample)
assert (label_sample["rule_check"] == label_sample["is_late"]).all()
assert set(work["is_late"].unique()) == {0, 1}
print("Label rule validated on sampled real orders.")


,order_id,order_delivered_customer_date,order_estimated_delivery_date,is_late,rule_check
1,3b697a20d9e427646d92567910af6d57,2016-10-26 14:02:13,2016-10-27,0,0
2,be5bc2f0da14d8071e2d45451ad119d9,2016-10-27 18:19:38,2016-11-07,0,0
3,a41c8759fbe7aab36ea07e038b2d4465,2016-11-03 10:58:07,2016-11-29,0,0
0,bfbd0f9bdef84302105ad712db648a6c,2016-11-09 07:47:38,2016-10-04,1,1
42,b0cfee6ce5503781be6004e0e31e4309,2016-11-29 15:23:44,2016-11-28,1,1
52,5cc475c7c03290048eb2e742cd64cb5e,2016-12-12 20:31:54,2016-11-24,1,1


Label rule validated on sampled real orders.


## 3. Check class distribution and imbalance


In [3]:
class_distribution = (work["is_late"].value_counts().rename_axis("is_late").reset_index(name="count"))
class_distribution["percentage"] = 100 * class_distribution["count"] / len(work)
class_distribution["label_name"] = class_distribution["is_late"].map({0: "on_time", 1: "late"})
display(class_distribution.sort_values("is_late"))
majority = class_distribution["count"].max()
minority = class_distribution["count"].min()
imbalance_ratio = majority / minority
print(f"Imbalance ratio (majority/minority): {imbalance_ratio:.2f}")


,is_late,count,percentage,label_name
0,0,88644,91.887633,on_time
1,1,7826,8.112367,late


Imbalance ratio (majority/minority): 11.33


## 4. Save the labeled artifact


In [4]:
work.to_parquet(output_path, index=False)
metadata = {"artifact": output_path.name, "grain": "one row per order_id", "rows": int(len(work)), "target": "is_late", "target_definition": "1 if actual delivery is after estimated delivery, otherwise 0", "excluded_rows": int(len(df) - len(work)), "excluded_rule": "Only delivered orders with both delivery dates are labeled"}
metadata_path.write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")
print(f"Saved {output_path}")
print(f"Saved {metadata_path}")


Saved C:\Users\HP\Desktop\MLOps\Task2\artifacts\tables\02_labeled_table.parquet
Saved C:\Users\HP\Desktop\MLOps\Task2\artifacts\reports\02_labeled_table_metadata.json


## Final completion check


In [5]:
reloaded = pd.read_parquet(output_path)
assert reloaded["order_id"].is_unique
assert reloaded["is_late"].notna().all()
assert set(reloaded["is_late"].unique()) == {0, 1}
print("ALL NOTEBOOK 2 CHECKS PASSED")


ALL NOTEBOOK 2 CHECKS PASSED
